# Processamento de imagens 2D com efeito pseudo-3D

## Seção Teórica

### Introdução

A síntese de novas visualizações a partir de uma única imagem 2D é um tema recorrente em computação visual, especialmente quando se busca produzir **sensação de profundidade** e **paralaxe** em cenários interativos com baixo custo computacional. Em vez de reconstruir explicitamente uma geometria 3D completa, abordagens práticas exploram **transformações geométricas (warping)** guiadas por alguma forma de informação de profundidade, real ou estimada.

Nesse contexto, a **distorção baseada em profundidade** e a *Depth-Image-Based Rendering* (DIBR) oferecem uma formulação amplamente utilizada para a síntese de pontos de vista virtuais: parte-se de uma imagem original e de um mapa de profundidade associado aos seus pixels e, em seguida, remapeiam-se amostras de cor de modo consistente com a variação de ponto de vista (FEHN, 2004). Um efeito perceptual típico desse processo é a paralaxe, particularmente visível em deslocamentos laterais modestos (MAO; CHEUNG; JI, 2014).

Esta seção teórica organiza os conceitos necessários para fundamentar a seção prática do notebook: (i) a noção de **distorção de imagens** como função de mapeamento entre planos; (ii) o papel de mapas de profundidade na síntese de novas visualizações; e (iii) a operacionalização do remapeamento com a função `cv2.remap` (OpenCV). Na parte prática, implementa-se uma versão simplificada do procedimento para simular um efeito pseudo‑3D com controle interativo, priorizando clareza metodológica e reprodutibilidade.

### Distorção de imagens

A distorção de imagens (*image warping*) é um procedimento recorrente em computação visual que consiste em aplicar uma **transformação geométrica** sobre imagens bidimensionais, estando presente em diversas aplicações (GUSTAFSSON, 1993).

De modo geral, uma distorção pode ser descrita por uma **função de mapeamento** entre a imagem original e uma imagem de destino, na qual as coordenadas de um ponto em um dos domínios determinam as coordenadas correspondentes no outro domínio (GUSTAFSSON, 1993).

Entre as diferentes classes de warping, destacam-se as técnicas **guiadas por profundidade**, nas quais um mapa de profundidade (real ou estimado) é utilizado para sintetizar variações de ponto de vista de uma câmera virtual (GUO et al., 2023).

Em termos conceituais, o warping guiado por profundidade é próximo da *Depth-Image-Based Rendering* (DIBR). A distinção prática, no escopo deste notebook, está no objetivo: busca-se produzir **uma nova imagem 2D** que simule mudanças de perspectiva (paralaxe), em vez de reconstruir explicitamente uma representação 3D completa.

### Renderização de imagem baseada em profundidade (DIBR)

A *Depth-Image-Based Rendering* (DIBR) consiste em sintetizar visualizações "virtuais" (novos pontos de vista) de uma imagem — estática ou em vídeo — a partir das informações de **cor** e **profundidade** associadas a cada pixel (FEHN, 2004).

A formulação clássica do processo pode ser descrita em duas etapas (FEHN, 2004):

- As amostras da imagem são posicionadas em um espaço 3D usando a profundidade de cada pixel.
- Em seguida, esses pontos 3D são projetados no plano de uma **câmera virtual** posicionada no ponto de vista desejado.

Mao, Cheung e Ji (2014) observam que um efeito característico do uso de DIBR é a **paralaxe**, particularmente perceptível em deslocamentos laterais (em torno do eixo *x*), ao passo que deslocamentos ao longo do eixo de profundidade (aproximar-se ou afastar-se) tendem a produzir uma variação perceptual menos evidente nesse tipo de síntese.

![Exemplo DIBR](imgs/DIBR-Mao-Cheung-Ji.png)

*Imagem 1: Exemplo de aplicação de DIBR em uma imagem (adaptado de Mao, Cheung e Ji).*

### Biblioteca OpenCV (cv2)

Para operacionalizar a distorção guiada por profundidade (como uma implementação simplificada inspirada em DIBR), este notebook utiliza a função `remap` da biblioteca OpenCV (`cv2` no Python). Essa função permite **remapear** (deslocar) os pixels da imagem de entrada para novas coordenadas na imagem de saída, realizando interpolação quando necessário (HUAMAN, *[s.d.]*).

Em termos de formulação, `remap` aplica uma função de mapeamento sobre as coordenadas *x* e *y*, o que possibilita inverter, redirecionar ou deformar a imagem conforme os mapas de transformação fornecidos (HUAMAN, *[s.d.]*).

Ao empregar `borderMode=cv2.BORDER_REPLICATE`, regiões que extrapolam os limites da imagem são preenchidas por **replicação das bordas**. Embora essa escolha evite áreas vazias, ela pode introduzir artefatos e reduzir a naturalidade da síntese. Em pipelines completos de DIBR, alternativas incluem estratégias de preenchimento e suavização, por exemplo com filtragem Gaussiana aplicada ao mapa de profundidade para reduzir descontinuidades (FEHN, 2004).

### Referências bibliográficas

**FEHN, Christoph**. *Depth-image-based rendering (DIBR), compression, and transmission for a new approach on 3D-TV*. Em: Stereoscopic Displays and Virtual Reality Systems XI. SPIE, 2004. p. 93-104.

**MAO, Yu; CHEUNG, Gene; JI, Yusheng**. *Image interpolation for DIBR view synthesis using graph Fourier transform*. Em: 2014 3DTV-Conference: The True Vision - Capture, Transmission and Display of 3D Video (3DTV-CON). Budapeste, Hungria, 2014. p. 1-4.

**HUAMAN, Ana**. *Remapping*. Documentação OpenCV. *[s.l.]*, *[s.d.]*. Disponível em: https://docs.opencv.org/4.x/d1/da0/tutorial_remap.html. Acesso em 28/04/2026.

**GUSTAFSSON, Andreas**. *Interactive Image Warping*. *[s.l.]*, 1993. p. 1-59.

**GUO, Mantang; et al.** *Content-aware Warping for View Synthesis*. *[s.l.]*, 2023. p. 1-18.

## Seção Prática

### Dependências

Antes de iniciar a execução do notebook, é importante garantir que todas as dependências dele estejam instaladas. 

Para isso, basta instalar as dependências descritas em `requirements.txt` ou executar a célula abaixo.

**IMPORTANTE:** Execute a célula abaixo somente **uma vez** nesse kernel.

In [1]:
import sys
import subprocess
from pathlib import Path

req_file = Path("requirements.txt")

if req_file.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(req_file)])
else:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "opencv-python",
            "numpy",
            "Pillow",
            "ipywidgets",
        ]
    )

print("Dependências instaladas neste kernel.")


Dependências instaladas neste kernel.


### Bibliotecas utilizadas

- **OpenCV (`cv2`)**: leitura de imagem, redimensionamento e (principalmente) o remapeamento de pixels com `cv2.remap`.
- **NumPy (`numpy`)**: criação das grades de coordenadas e cálculos matriciais do mapa de profundidade e dos deslocamentos.
- **PIL (`Pillow`)** + **IPython display**: conversão para RGB e exibição da imagem no notebook.
- **ipywidgets**: criação da interface interativa com sliders e `widgets.interact`.

In [2]:
import cv2
import numpy as np
from IPython.display import display
from PIL import Image
import ipywidgets as widgets

### Carregamento da imagem

 notebook define um caminho em `caminho_da_imagem` e tenta ler com o comando `img = cv2.imread(caminho_da_imagem)`.

In [ ]:
# IMPORTANTE: Altere o caminho da imagem para o local onde a sua imagem está armazenada do seu computador.
caminho_da_imagem = 'C:\\Users\\vinic\\OneDrive\\Documentos\\ReposMackenzie\\imagens-pseudo3D\\inputs\\cachorro-gravata.jpg' 
img = cv2.imread(caminho_da_imagem)

if img is None:
    raise ValueError("A imagem não foi encontrada.")

### Ajuste de tamanho (redimensionamento)

Depois de obter `altura` e `largura`, o notebook limita a imagem a:

- `max_width = 600`
- `max_height = 500`

Se a imagem for maior que esses limites, calcula uma **escala** e redimensiona mantendo a proporção:

- `escala = min(max_width / largura, max_height / altura)`
- `cv2.resize(...)`

In [ ]:
altura, largura = img.shape[:2]

max_width, max_height = 600, 500
if largura > max_width or altura > max_height:
    escala = min(max_width / largura, max_height / altura)
    img = cv2.resize(img, (int(largura * escala), int(altura * escala)))
    altura, largura = img.shape[:2]

### Criação de um “mapa de profundidade” sintético

Como não há um depth map real (estimado por rede neural, estéreo, etc.), o notebook cria um mapa **artificial** baseado na distância ao centro da imagem:

1. cria grades de coordenadas:
   - `y, x = np.mgrid[0:altura, 0:largura]`
2. calcula o centro:
   - `centro_x = largura / 2`, `centro_y = altura / 2`
3. define um mapa com formato “colina” (tipo gaussiano/exponencial radial):
   - `mapa_profundidade = exp(-dist2 / escala)`

Em seguida, converte esse mapa em deslocamento máximo de pixels:

- `forca_maxima = 30`
- `profundidade_calculada = mapa_profundidade * forca_maxima`

Na prática, isso faz o **centro deslocar mais** e as bordas deslocarem menos (ou vice‑versa, dependendo do sinal do controle), produzindo a sensação de profundidade ao mover.

In [13]:
y, x = np.mgrid[0:altura, 0:largura]
centro_x, centro_y = largura / 2, altura / 2
mapa_profundidade = np.exp(-((x - centro_x)**2 + (y - centro_y)**2) / (0.1 * (largura**2 + altura**2)))

forca_maxima = 30 
profundidade_calculada = mapa_profundidade * forca_maxima

print(f"Dimensões da imagem: {largura}x{altura}\n")

Dimensões da imagem: 300x168



### Interação, efeito pseudo-3D e resultado final

O notebook define a função `exibir_simulacao(...)` com dois parâmetros controlados por sliders:

- `dx`: varia de `-1.0` a `+1.0` (movimento horizontal)
- `dy`: varia de `-1.0` a `+1.0` (movimento vertical)

Ela é conectada à UI com:

- `widgets.interact(exibir_simulacao)`

Cada alteração no slider chama a função novamente e atualiza a imagem mostrada.

Dentro de `exibir_simulacao`, o notebook calcula mapas de remapeamento (para cada pixel de saída, de onde vem o pixel na imagem original):

- `mapa_x = x - profundidade_calculada * dx`
- `mapa_y = y - profundidade_calculada * dy`

E aplica o remapeamento:

- `imagem_deslocada = cv2.remap(img, mapa_x, mapa_y, interpolation=..., borderMode=...)`

Configurações importantes:

- **`interpolation=cv2.INTER_LINEAR`**: suaviza a amostragem ao deslocar.
- **`borderMode=cv2.BORDER_REPLICATE`**: quando o deslocamento “puxa” pixels de fora da imagem, repete a borda para evitar áreas pretas.

Esse processo é um uso típico de **DIBR (Depth‑Image‑Based Rendering)** simplificado: em vez de renderizar um modelo 3D, desloca-se a imagem conforme uma profundidade por pixel.

Como o OpenCV trabalha em **BGR**, o notebook converte para **RGB** antes de mostrar:

- `cv2.cvtColor(..., cv2.COLOR_BGR2RGB)`
- `Image.fromarray(...)`
- `display(...)`

Ele também imprime os valores atuais do “joystick” (`dx`, `dy`) para feedback.

In [ ]:
def exibir_simulacao(dx=widgets.FloatSlider(min=-1.0, max=1.0, step=0.05, value=0.0, description='X (←→):'),
                     dy=widgets.FloatSlider(min=-1.0, max=1.0, step=0.05, value=0.0, description='Y (↑↓):')):
    
    # Calcula os mapas de deslocamento
    mapa_x = np.float32(x - profundidade_calculada * dx)
    mapa_y = np.float32(y - profundidade_calculada * dy)
    
    imagem_deslocada = cv2.remap(img, mapa_x, mapa_y, 
                                 interpolation=cv2.INTER_LINEAR, 
                                 borderMode=cv2.BORDER_REPLICATE)
    
    # Converte para RGB
    img_rgb = cv2.cvtColor(imagem_deslocada, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)

    display(img_pil)
    
    # Mostra informações do joystick
    print(f"Joystick → X: {dx:+.2f} | Y: {dy:+.2f}")

# Cria a interface interativa com @interact
widgets.interact(exibir_simulacao)